## Fine-Tuning with SFT + LoRA for Qwen

This notebook demonstrates how to perform Supervised Fine-Tuning (SFT) on the Qwen model using Parameter-Efficient Fine-Tuning (PEFT) with LoRA and QLoRA.

### Workflow:
1.  **Setup**: Install required libraries and configure paths.
2.  **Data Preparation**: Load the dataset and format it for SFT. Since full reasoning is not available, we create synthetic completions to teach the model the output format.
3.  **Training**: Fine-tune the model using `SFTTrainer`, 4-bit quantization (QLoRA), and LoRA.
4.  **Merge & Save**: Merge the trained LoRA adapters into the base model and save the final fine-tuned model.
5.  **Inference**: Load the merged model with `vLLM` for high-throughput generation.
6.  **Evaluation**: Score the generated responses against the ground truth answers.

### DSMLP Setup
First, run this in your DSMLP terminal:
```bash
launch-sp26-cuda128.sh -l gpu-class=medium -W CSE151B_SP26_A00 -g 1 -c 8 -m 32 -v a30
```

In [1]:
!pip install -q "transformers==4.40.1" "datasets==2.19.0" "accelerate==0.29.3" "bitsandbytes==0.45.5" "peft==0.10.0" "trl==0.8.6" "vllm==0.19.1"

ERROR: Cannot install peft==0.10.0, transformers==4.40.1, trl==0.8.6 and vllm==0.19.1 because these package versions have conflicting dependencies.


ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [2]:
# Verify library versions
import vllm
import trl
import peft
import transformers
print(f"vLLM version: {vllm.__version__}")
print(f"transformers version: {transformers.__version__}")
print(f"peft version: {peft.__version__}")
print(f"trl version: {trl.__version__}")

vLLM version: 0.19.1
transformers version: 4.57.6
peft version: 0.10.0
trl version: 0.8.6


In [7]:
import json
import os
import re
import sys
import gc
import torch
from pathlib import Path
from typing import Optional, List

# ── Configuration ─────────────────────────────────────────────────────────────
GPU_ID             = "0"
BASE_MODEL_ID      = "Qwen/Qwen3-4B-Thinking-2507" # Using the same model as baseline
DATA_PATH          = "data/public.jsonl"      # Adjusted path relative to notebook
ADAPTER_PATH       = "results/sft_lora_adapter"
MERGED_MODEL_PATH  = "results/qwen_sft_merged"
OUTPUT_PATH        = "results/sft_results.jsonl"
MAX_SEQ_LENGTH     = 2048 # Further reduced for GPUs with ~20GB VRAM
MAX_TOKENS_INFER   = 32768 # Max tokens for vLLM generation, matching baseline

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

from datasets import Dataset
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer
from vllm import LLM, SamplingParams
from tqdm import tqdm

In [8]:
# Check GPU status
!nvidia-smi

Tue May 26 07:40:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 PCIe               On  |   00000000:C3:00.0 Off |                   On |
| N/A   39C    P0            108W /  350W |                  N/A   |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----

|  0    5   0   0  |              17MiB / 20096MiB    | 14      0 |  1   0    1    0    1 |
|                  |               0MiB / 12370MiB    |           |                       |
+------------------+----------------------------------+-----------+-----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|


|  No running processes found                                                             |
+-----------------------------------------------------------------------------------------+


## Part 1: Data Preparation for SFT

In [9]:
# Load tokenizer early for data preparation
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded.")

Tokenizer loaded.


In [10]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

def create_sft_dataset(data: List[dict], tokenizer: AutoTokenizer) -> Dataset:
    """Formats raw data into a dataset for SFT, with each entry as a formatted string."""
    formatted_texts = []
    for item in data:
        system, user = build_prompt(item["question"], item.get("options"))
        
        # NOTE: The public dataset does not contain step-by-step reasoning.
        # We create a synthetic completion that only includes the final answer in the required format.
        # This fine-tunes the model to follow output formatting instructions.
        if item.get("options"):
            assistant_response = f"The final answer is \\boxed{{{item['answer']}}}."
        else:
            gold_answer = item['answer']
            if isinstance(gold_answer, list):
                gold_answer = gold_answer[0] # Use the first answer for simplicity in training
            assistant_response = f"The final answer is \\boxed{{{gold_answer}}}."

        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant_response}
        ]
        
        # Apply the chat template to create a single string for SFT
        # add_generation_prompt=False because we are providing the full conversation
        formatted_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        formatted_texts.append({"text": formatted_text})
        
    return Dataset.from_list(formatted_texts)

# Load data and create SFT dataset
raw_data = [json.loads(line) for line in open(DATA_PATH)]
sft_dataset = create_sft_dataset(raw_data, tokenizer)

print(f"Loaded and formatted {len(sft_dataset)} samples for SFT.")
print("\n── SFT Sample (formatted text) ──")
print(sft_dataset[0]['text'])

Loaded and formatted 1126 samples for SFT.

── SFT Sample (formatted text) ──
<|im_start|>system
You are an expert mathematician. Solve the problem step-by-step. Put your final answer inside \boxed{}. If the problem has multiple sub-answers, separate them by commas inside a single \boxed{}, e.g. \boxed{3, 7}.<|im_end|>
<|im_start|>user
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]<|im_end|>
<|im_start|>assistant
<think>

</think>

The final answer is \boxed{325*(1+325)}.<|im_end|>



## Part 2: QLoRA Fine-Tuning

In [11]:
# QLoRA configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load base model with quantization
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# LoRA configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # For Qwen2, target modules are typically these. Adjust if using a different model.
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

# Training arguments
training_args = TrainingArguments(
    output_dir=ADAPTER_PATH,
    num_train_epochs=1,
    per_device_train_batch_size=1,          # Reduced for memory constraints
    gradient_accumulation_steps=8,          # Increased to maintain effective batch size
    gradient_checkpointing=True,            # Enable gradient checkpointing to save memory
    gradient_checkpointing_kwargs={"use_reentrant": False}, # Recommended setting
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    fp16=False, # bf16 is enabled by default with torch_dtype=torch.bfloat16
    bf16=True,
    report_to="none",
)

# Initialize SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=sft_dataset,
    peft_config=peft_config,
    dataset_text_field="text", # Specify the column containing the formatted text
    dataset_num_proc=4,
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_args,
)

# Start training
print("Starting SFT with QLoRA...")
trainer.train()
trainer.save_model(ADAPTER_PATH)
print(f"Training complete. LoRA adapters saved to {ADAPTER_PATH}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Map (num_proc=4):   0%|          | 0/1126 [00:00<?, ? examples/s]

/home/kik012/.local/lib/python3.13/site-packages/trl/trainer/sft_trainer.py:323: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting SFT with QLoRA...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,1.288000
20,0.732700
30,0.647400
40,0.582400
50,0.646000
60,0.677000
70,0.659700
80,0.668700
90,0.640200
100,0.601400


Training complete. LoRA adapters saved to results/sft_lora_adapter


## Part 3: Merge Adapters and Save Model

In [12]:
# Clean up memory before loading full model
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

print("Loading base model for merging (on CPU to save VRAM)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cpu", # Load on CPU to prevent OOM during merge
    trust_remote_code=True,
)

print(f"Loading PEFT model from {ADAPTER_PATH}...")
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

print("Merging LoRA adapters...")
merged_model = peft_model.merge_and_unload()

print(f"Saving merged model to {MERGED_MODEL_PATH}...")
merged_model.save_pretrained(MERGED_MODEL_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_MODEL_PATH)

print("Model merging and saving complete.")

Loading base model for merging (on CPU to save VRAM)...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading PEFT model from results/sft_lora_adapter...


Merging LoRA adapters...


Saving merged model to results/qwen_sft_merged...


Model merging and saving complete.


## Part 4: Inference with the Fine-Tuned Model

In [13]:
# Clean up memory again before loading with vLLM
# Use try-except blocks to avoid NameError if the previous cell failed
try:
    del base_model
except NameError:
    pass
try:
    del peft_model
except NameError:
    pass
try:
    del merged_model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

print(f"Loading fine-tuned model from {MERGED_MODEL_PATH} with vLLM...")
# NOTE: Parameters are adjusted to be more conservative for a ~20GB GPU.
# We also increase max_num_seqs to allow for batching and faster inference.
llm = LLM(
    model=MERGED_MODEL_PATH,
    dtype="bfloat16",
    gpu_memory_utilization=0.65, # Reduced to leave more memory headroom
    max_model_len=MAX_SEQ_LENGTH, # Use the same length as training (2048)
    trust_remote_code=True,
    max_num_seqs=8,              # Reduced batch size to lower memory usage
    max_num_batched_tokens=MAX_SEQ_LENGTH * 8,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS_INFER, # 32768
    temperature=0.7,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Fine-tuned model loaded for inference.")

Loading fine-tuned model from results/qwen_sft_merged with vLLM...
INFO 05-26 08:06:10 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.65, 'max_num_batched_tokens': 16384, 'max_num_seqs': 8, 'disable_log_stats': True, 'model': 'results/qwen_sft_merged'}


INFO 05-26 08:06:10 [model.py:549] Resolved architecture: Qwen3ForCausalLM


INFO 05-26 08:06:10 [model.py:1678] Using max model len 2048


INFO 05-26 08:06:10 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.


INFO 05-26 08:06:10 [vllm.py:790] Asynchronous scheduling is enabled.


...

(EngineCore pid=10625) 
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):   0%|          | 0/5 [00:00<?, ?it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  80%|████████  | 4/5 [00:00<00:00, 30.32it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 5/5 [00:00<00:00, 29.69it/s]
(EngineCore pid=10625) 
Capturing CUDA graphs (decode, FULL):   0%|          | 0/4 [00:00<?, ?it/s]


Capturing CUDA graphs (decode, FULL): 100%|██████████| 4/4 [00:00<00:00, 33.63it/s]


(EngineCore pid=10625) INFO 05-26 08:07:05 [gpu_model_runner.py:6046] Graph capturing finished in 2 secs, took 0.07 GiB
(EngineCore pid=10625) INFO 05-26 08:07:05 [gpu_worker.py:597] CUDA graph pool memory: 0.07 GiB (actual), 0.06 GiB (estimated), difference: 0.01 GiB (13.9%).
(EngineCore pid=10625) INFO 05-26 08:07:05 [core.py:283] init engine (profile, create kv cache, warmup model) took 26.82 seconds


(EngineCore pid=10625) The tokenizer you are loading from 'results/qwen_sft_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


(EngineCore pid=10625) INFO 05-26 08:07:05 [vllm.py:790] Asynchronous scheduling is enabled.
Fine-tuned model loaded for inference.


In [28]:
# Build prompts for the entire dataset
prompts = []
for item in raw_data[:100]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={raw_data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 100 questions...


Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


── Response 0 (id=0) ──
</think>

The final answer is \boxed{105425}. 

── Response 1 (id=1) ──
</think>

The final answer is \boxed{I}. 

── Response 2 (id=2) ──
</think>

The final answer is \boxed{139.815580158016}. The final answer is \boxed{1.71947783021065}. 


## Part 5: Scoring and Evaluation

In [23]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

# Load Judger for free-form scoring
sys.path.insert(0, "..") # Adjust path to find judger.py
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(raw_data, responses), total=len(raw_data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")


Scoring:   0%|          | 0/1126 [00:00<?, ?it/s]


Scoring:   1%|          | 8/1126 [00:00<00:24, 45.44it/s]


Scoring:   1%|          | 13/1126 [00:00<00:25, 44.15it/s]


Scoring:   2%|▏         | 27/1126 [00:00<00:15, 73.23it/s]


Scoring:   3%|▎         | 35/1126 [00:00<00:31, 34.78it/s]

...


Scoring:   6%|▋         | 71/1126 [00:01<00:22, 46.02it/s]


Scoring:   7%|▋         | 78/1126 [00:01<00:24, 41.94it/s]


Scoring:   8%|▊         | 85/1126 [00:01<00:23, 44.81it/s]


Scoring:   9%|▉         | 100/1126 [00:01<00:18, 54.74it/s]

Scoring complete. 100 results.


In [24]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS (SFT MODEL)")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS (SFT MODEL)
  MCQ        :   13 /   38  (34.21%)
  Free-form  :    6 /   62  (9.68%)
  Overall    :   19 /  100  (19.00%)


In [20]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 10 records to results/sft_results.jsonl


## Part 6: Generate Submission for Private Test Set

The following cell will:
1. Load the `private.jsonl` file.
2. Generate responses for all questions in the private set using the fine-tuned model.
3. Save the results to `sft_submission.csv` in the required format (`id,response`).

**Note**: This will run inference on the entire private dataset and may take some time, but it will be much faster now with batching enabled.

In [25]:
import csv

# --- Configuration for Private Set ---
PRIVATE_DATA_PATH = "data/private.jsonl"
SUBMISSION_CSV_PATH = "sft_submission.csv"

# --- Check if private data exists ---
private_data_path = Path(PRIVATE_DATA_PATH)
if not private_data_path.exists():
    print(f"Private data file not found at {PRIVATE_DATA_PATH}. Skipping submission generation.")
else:
    print(f"Loading private data from {PRIVATE_DATA_PATH}...")
    private_data = [json.loads(line) for line in open(private_data_path)]
    print(f"Loaded {len(private_data)} questions from the private set.")

    # --- Build prompts for the private set ---
    private_prompts = []
    for item in private_data:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )
        private_prompts.append(prompt_text)

    # --- Generate responses ---
    print(f"Generating responses for {len(private_prompts)} private questions...")
    private_outputs = llm.generate(private_prompts, sampling_params=sampling_params)
    private_responses = [out.outputs[0].text.strip() for out in private_outputs]

    # --- Save to submission.csv ---
    print(f"Saving submission file to {SUBMISSION_CSV_PATH}...")
    with open(SUBMISSION_CSV_PATH, "w", newline="") as f:
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        writer.writerow(["id", "response"])
        for item, response in zip(private_data, private_responses):
            writer.writerow([item["id"], response])

    print(f"Submission file '{SUBMISSION_CSV_PATH}' created successfully.")

Loading private data from data/private.jsonl...
Loaded 943 questions from the private set.
Generating responses for 943 private questions...


Rendering prompts:   0%|          | 0/943 [00:00<?, ?it/s]

VLLMValidationError: This model's maximum context length is 2048 tokens. However, you requested 0 output tokens and your prompt contains at least 2049 input tokens, for a total of at least 2049 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=2049)